# Chunker
Splits every real-data experiment document into 4 chunks (investigation+fuel, conditions, results, risk+summary) and saves `rag/embeddings/processed/chunks/chunks.json`.

In [1]:
from pathlib import Path
import sys

# Works on every machine: find the project root (the folder that contains rag/pipeline.py)
PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "rag" / "pipeline.py").exists()
)
RAG_DIR = PROJECT_ROOT / "rag"

import re
import json


# ============================================================
# Configuration
# ============================================================

KNOWLEDGE_BASE = PROJECT_ROOT / "knowledge_base" / "experiments"

# The retrievers, embedder and metadata filter all read this file.
OUTPUT_DIR = RAG_DIR / "embeddings" / "processed" / "chunks"
OUTPUT_FILE = OUTPUT_DIR / "chunks.json"


# ============================================================
# Parse YAML-like front matter
# ============================================================

def _convert(value):
    if value == "":
        return None
    if value in ("true", "false"):
        return value == "true"
    try:
        return float(value) if "." in value else int(value)
    except ValueError:
        return value


def parse_metadata(content):
    metadata = {}
    match = re.match(r"^---\s*\n(.*?)\n---\s*\n", content, re.DOTALL)
    if not match:
        return metadata, content
    for line in match.group(1).splitlines():
        if ":" not in line:
            continue
        key, value = line.split(":", 1)
        metadata[key.strip()] = _convert(value.strip())
    return metadata, content[match.end():].strip()


# ============================================================
# Extract sections
# ============================================================

def extract_sections(document_text):
    sections = {}
    matches = list(re.finditer(r"^##\s+(.+?)\s*$", document_text, re.MULTILINE))
    for i, match in enumerate(matches):
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(document_text)
        sections[match.group(1).strip()] = document_text[start:end].strip()
    return sections


# ============================================================
# Create chunks for one experiment
# ============================================================

CHUNK_LAYOUT = [
    ("investigation_fuel", ["Investigation", "Fuel"]),
    ("experimental_conditions", ["Experimental Conditions"]),
    ("observed_results", ["Observed Results"]),
    ("risk_summary", ["Fire Risk", "Experimental Summary"]),
]


def create_chunks(content):
    metadata, document_text = parse_metadata(content)
    sections = extract_sections(document_text)
    experiment_id = metadata.get("experiment_id", "unknown")

    # Dataset-level summary documents (statistics over all 179 tests) = one chunk each.
    if metadata.get("doc_type") in ("summary", "nasa_psi"):
        return [{
            "text": document_text.strip(),
            "metadata": {**metadata, "chunk_type": "dataset_summary"
                         if metadata.get("doc_type") == "summary" else "nasa_psi_document"},
        }]

    # One-line header repeated in every chunk so each chunk is self-describing.
    header = (f"# Experiment {experiment_id} — {metadata.get('fuel_mix')}, "
              f"O2 {metadata.get('oxygen_before_pct')}%, {metadata.get('pressure_kpa')} kPa")
    chunks = []
    for chunk_type, names in CHUNK_LAYOUT:
        parts = [f"## {n}\n\n{sections[n]}" for n in names if sections.get(n)]
        if not parts:
            continue
        chunks.append({
            "text": header + "\n\n" + "\n\n".join(parts),
            "metadata": {**metadata, "chunk_type": chunk_type},
        })
    return chunks


def chunk_knowledge_base():
    all_chunks = []
    files = sorted(KNOWLEDGE_BASE.glob("*.md"))
    print(f"Found {len(files)} Markdown files.")
    for file_path in files:
        content = file_path.read_text(encoding="utf-8")
        for chunk in create_chunks(content):
            chunk["metadata"]["source"] = file_path.name
            all_chunks.append(chunk)
    return all_chunks


def save_chunks(chunks):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for index, chunk in enumerate(chunks):
        chunk["chunk_id"] = index          # stable IDs = row index in the FAISS index
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)
    print("\nChunks saved to:")
    print(OUTPUT_FILE.resolve())


chunks = chunk_knowledge_base()

print("\n================================")
print("CHUNKING COMPLETE")
print("================================")
print(f"Total chunks: {len(chunks)}")
n_tests = sum(1 for c in chunks if c["metadata"]["experiment_id"].startswith("CFI-")) // 4
n_summaries = sum(1 for c in chunks if c["metadata"].get("doc_type") == "summary")
n_psi = sum(1 for c in chunks if c["metadata"].get("doc_type") == "nasa_psi")
print(f"Real tests: {n_tests} (4 chunks each) + {n_summaries} dataset summaries + {n_psi} NASA PSI documents")
assert n_tests == 179, "expected 179 real tests"
assert all(c["metadata"]["experiment_id"].startswith(("CFI-", "SUMMARY-", "PSI-DOC-")) for c in chunks)

save_chunks(chunks)

print("\nFirst chunk:")
print("--------------------------------")
print(chunks[0]["text"])
print("\nMetadata:")
print("--------------------------------")
print(chunks[0]["metadata"])

Found 234 Markdown files.

CHUNKING COMPLETE
Total chunks: 771
Real tests: 179 (4 chunks each) + 6 dataset summaries + 49 NASA PSI documents

Chunks saved to:
C:\Users\Ahmed\.zcode\workspace\default\Nasa Rag SYS\rag\embeddings\processed\chunks\chunks.json

First chunk:
--------------------------------
# Experiment CFI-21088-A1 — ethane, O2 39.39%, 94.4 kPa

## Investigation

- **Source Investigation:** PSI-159 (CFI-G) — NASA Cool Flames Investigation with Gases, normal (fuel-into-oxidizer) gaseous diffusion flames
- **Platform:** International Space Station (microgravity)
- **PSI-159 Test ID:** PSI159-21088A1
- **Test Date:** 2021-03-29

## Fuel

- **Fuel:** ethane
- **Fuel Mix:** ethane
- **Fuel Mole Fraction (Xf):** 1.0
- **Fuel Bottle Concentration:** 100.0 %

Metadata:
--------------------------------
{'experiment_id': 'CFI-21088-A1', 'psi159_id': 'PSI159-21088A1', 'source_investigation': 'PSI-159 (CFI-G)', 'test_date': '2021-03-29', 'fuel': 'ethane', 'fuel_mix': 'ethane', 'oxygen_